<pre>
1. Download the data from <a href='https://drive.google.com/file/d/15dCNcmKskcFVjs7R0ElQkR61Ex53uJpM/view?usp=sharing'>here</a>

2. Code the model to classify data like below image

<img src='https://i.imgur.com/33ptOFy.png'>

3. Write your own callback function, that has to print the micro F1 score and AUC score after each epoch.

4. Save your model at every epoch if your validation accuracy is improved from previous epoch. 

5. you have to decay learning based on below conditions 
        Cond1. If your validation accuracy at that epoch is less than previous epoch accuracy, you have to decrese the
               learning rate by 10%. 
        Cond2. For every 3rd epoch, decay your learning rate by 5%.
        
6. If you are getting any NaN values(either weigths or loss) while training, you have to terminate your training. 

7. You have to stop the training if your validation accuracy is not increased in last 2 epochs.

8. Use tensorboard for every model and analyse your gradients. (you need to upload the screenshots for each model for evaluation)

9. use cross entropy as loss function

10. Try the architecture params as given below. 
</pre>

<pre>
<b>Model-1</b>
<pre>
1. Use tanh as an activation for every layer except output layer.
2. use SGD with momentum as optimizer.
3. use RandomUniform(0,1) as initilizer.
3. Analyze your output and training process. 
</pre>
</pre>
<pre>
<b>Model-2</b>
<pre>
1. Use relu as an activation for every layer except output layer.
2. use SGD with momentum as optimizer.
3. use RandomUniform(0,1) as initilizer.
3. Analyze your output and training process. 
</pre>
</pre>
<pre>
<b>Model-3</b>
<pre>
1. Use relu as an activation for every layer except output layer.
2. use SGD with momentum as optimizer.
3. use he_uniform() as initilizer.
3. Analyze your output and training process. 
</pre>
</pre>
<pre>
<b>Model-4</b>
<pre>
1. Try with any values to get better accuracy/f1 score.  
</pre>
</pre>

In [33]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [34]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [35]:
import datetime
from tensorflow.keras.layers import Dense,Input,Activation
from tensorflow.keras.models import Model
import random as rn
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import LearningRateScheduler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc,f1_score
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import TerminateOnNaN

In [36]:
data = pd.read_csv("data.csv")
data.shape
X = data.drop('label',axis=1)
Y = data['label']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
      X,Y , test_size=0.2, stratify=Y)
Y_train = tf.keras.utils.to_categorical(y_train, 2) 
Y_test = tf.keras.utils.to_categorical(y_test, 2)

# Logic of custom callback for calculating auc score and micro F1 score

In [38]:
class Custom_f1_auc(tf.keras.callbacks.Callback):
    
    def __init__(self, X_val,Y_val): 
        super(Custom_f1_auc, self).__init__()
        self.X = X_val
        self.Y = Y_val
        #print(self.X,self.Y)
    
    def on_train_begin(self, logs={}):
        ## on begin of training, we are creating a instance varible called history
        ## it is a dict with keys [loss, acc, val_loss, val_acc]
        self.history={'loss': [],'acc': [],'val_loss': [],'val_acc': []}
        self.cust_accuracy = []
        self.f1_score = []
        
    def on_epoch_end(self, epoch, logs={}):
        prob_score_of_y = self.model.predict(self.X)
        original_y = np.array(self.Y)
        calculated_y = []
        fpr, tpr, thresholds = roc_curve(original_y[:,1], prob_score_of_y[:,1])
        auc_score = auc(fpr,tpr)
        self.cust_accuracy.append(auc_score)
        for value in prob_score_of_y[:,1]:
            if value >=0.5:
                calculated_y.append(1)
            else:
                calculated_y.append(0)
        classified_y = np.array(calculated_y)
        micro_f1_score =f1_score(original_y[:,1], classified_y, average='micro') 
        self.f1_score.append(micro_f1_score)
        print("MICRO F1 SCORE",micro_f1_score)
        print("auc score",auc_score)
                
                

# Logic of learning rate change for every 3rd epoc

In [39]:
def changeLearningRate(epoch,lr):
    print("no of ecpoch",epoch)
    print("learning rate",lr)
    if (epoch)%3==0:
        lr= lr*0.95
    return lr    

# Saving Model

In [51]:

filepath="model_save\\weights-{epoch:02d}-{val_acc:.4f}.hdf5"
checkpoint = ModelCheckpoint(filepath=filepath, monitor='val_loss',  verbose=1, save_best_only=True, mode='auto')

# For every 3rd epoch, decay your learning rate by 5%.

In [52]:
lrschedule = LearningRateScheduler(changeLearningRate, verbose=1)

# If your validation accuracy at that epoch is less than previous epoch accuracy, you have to decrese the learning rate by 10%

In [53]:
lrschedule_10_percent = ReduceLROnPlateau(monitor='val_acc', factor=0.9, patience=1, verbose=1, mode='auto')

# Terminate on nan

In [54]:
nan_termination = TerminateOnNaN()

# Early stopping 

In [77]:
#earlystop = EarlyStopping(monitor='val_acc', min_delta=0.35, patience=1, verbose=1)
earlystop = EarlyStopping(monitor='val_loss', min_delta=0.01, patience=2, verbose=1)

# TensorBoard

In [78]:
log_dir="logs\\fit\\" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir,histogram_freq=1, write_graph=True,write_grads=True)

# Custom call back

In [79]:
custom_auc_f1_score = Custom_f1_auc(X_test,Y_test)

# List of all call back

In [80]:
all_call_back = [custom_auc_f1_score,checkpoint,lrschedule,lrschedule_10_percent,nan_termination,earlystop,tensorboard_callback]

In [81]:
#!kill 7924

In [82]:
# %load_ext tensorboard
# %tensorboard --logdir logs/fit/20

# Model1

In [84]:
#Input layer
input_layer = Input(shape=(2,))
#Dense hidden layer
model1_layer1 = Dense(20,activation='tanh',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=30))(input_layer)
model1_layer2 = Dense(25,activation='tanh',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=20))(model1_layer1)
model1_layer3 = Dense(30,activation='tanh',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=10))(model1_layer2)
model1_layer4 = Dense(35,activation='tanh',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=40))(model1_layer3)
model1_layer5= Dense(40,activation='tanh',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=50))(model1_layer4)
#output layer
output = Dense(2,activation='softmax',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=60))(model1_layer5)
#Creating a model
model = Model(inputs=input_layer,outputs=output)


#Callbacks
#history_own = LossHistory()

optimizer = tf.keras.optimizers.SGD(learning_rate=0.001,momentum=0.9)

model.compile(optimizer=optimizer, loss='categorical_crossentropy',metrics=['acc',"AUC"])

model.fit(X_train,Y_train,epochs=10, validation_data=(X_test,Y_test), batch_size=16, callbacks=all_call_back)


no of ecpoch 0
learning rate 0.0010000000474974513

Epoch 00001: LearningRateScheduler reducing learning rate to 0.0009500000451225787.
Epoch 1/10
 996/1000 [============================>.] - ETA: 0s - loss: 0.6960 - acc: 0.5036 - auc: 0.5072MICRO F1 SCORE 0.5155
auc score 0.515683125

Epoch 00001: val_loss did not improve from 0.69315
1000/1000 [==============================] - 6s 6ms/step - loss: 0.6960 - acc: 0.5038 - auc: 0.5076 - val_loss: 0.6964 - val_acc: 0.5155 - val_auc: 0.5177 - lr: 9.5000e-04
no of ecpoch 1
learning rate 0.0009500000160187483

Epoch 00002: LearningRateScheduler reducing learning rate to 0.0009500000160187483.
Epoch 2/10
 996/1000 [============================>.] - ETA: 0s - loss: 0.6910 - acc: 0.5262 - auc: 0.5391MICRO F1 SCORE 0.549
auc score 0.5490243749999999

Epoch 00002: val_loss improved from 0.69315 to 0.68429, saving model to model_save\weights-02-0.5490.hdf5
1000/1000 [==============================] - 5s 5ms/step - loss: 0.6910 - acc: 0.5263 - auc

In [69]:
#!kill 17140

In [2]:
#%tensorboard --logdir ./logs/fit/

# Model2

In [9]:
input_layer = Input(shape=(2,))
#Dense hidden layer
model1_layer1 = Dense(20,activation='relu',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=30))(input_layer)
model1_layer2 = Dense(20,activation='relu',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=20))(model1_layer1)
model1_layer3 = Dense(25,activation='relu',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=10))(model1_layer2)
model1_layer4 = Dense(30,activation='relu',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=40))(model1_layer3)
model1_layer5= Dense(30,activation='relu',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=50))(model1_layer4)
#output layer
output = Dense(2,activation='softmax',kernel_initializer=tf.keras.initializers.RandomUniform(minval=0., maxval=1.,seed=60))(model1_layer5)
#Creating a model
model = Model(inputs=input_layer,outputs=output)


#Callbacks
#history_own = LossHistory()

optimizer = tf.keras.optimizers.SGD(learning_rate=0.001,momentum=0.9)

model.compile(optimizer=optimizer, loss='categorical_crossentropy',metrics=['acc',"AUC"])

model.fit(X_train,Y_train,epochs=10, validation_data=(X_test,Y_test), batch_size=16, callbacks=all_call_back)


no of ecpoch 0
learning rate 0.0010000000474974513

Epoch 00001: LearningRateScheduler reducing learning rate to 0.0009500000451225787.
Epoch 1/10
Instructions for updating:
If using Keras pass *_constraint arguments to layers.
 998/1000 [============================>.] - ETA: 0s - loss: 133.6563 - acc: 0.4917 - auc: 0.4964MICRO F1 SCORE 0.3333333333333333
auc score 0.5

Epoch 00001: val_loss improved from inf to 0.69321, saving model to model_save\weights-01-0.5000.hdf5
1000/1000 [==============================] - 6s 6ms/step - loss: 133.3904 - acc: 0.4919 - auc: 0.4966 - val_loss: 0.6932 - val_acc: 0.5000 - val_auc: 0.5000 - lr: 9.5000e-04
no of ecpoch 1
learning rate 0.0009500000160187483

Epoch 00002: LearningRateScheduler reducing learning rate to 0.0009500000160187483.
Epoch 2/10
 992/1000 [============================>.] - ETA: 0s - loss: 0.6932 - acc: 0.5042 - auc: 0.5000MICRO F1 SCORE 0.3333333333333333
auc score 0.5

Epoch 00002: val_loss did not improve from 0.69321

Epoch 0

In [3]:
#!kill 1714

In [4]:
#%tensorboard --logdir ./logs/fit/

# Model3

In [13]:
input_layer = Input(shape=(2,))
#Dense hidden layer
model1_layer1 = Dense(20,activation='relu',kernel_initializer=tf.keras.initializers.he_uniform(seed=30))(input_layer)
model1_layer2 = Dense(20,activation='relu',kernel_initializer=tf.keras.initializers.he_uniform(seed=20))(model1_layer1)
model1_layer3 = Dense(25,activation='relu',kernel_initializer=tf.keras.initializers.he_uniform(seed=10))(model1_layer2)
model1_layer4 = Dense(30,activation='relu',kernel_initializer=tf.keras.initializers.he_uniform(seed=40))(model1_layer3)
model1_layer5= Dense(30,activation='relu',kernel_initializer=tf.keras.initializers.he_uniform(seed=50))(model1_layer4)
#output layer
output = Dense(2,activation='softmax',kernel_initializer=tf.keras.initializers.he_uniform(seed=60))(model1_layer5)
#Creating a model
model = Model(inputs=input_layer,outputs=output)


#Callbacks
#history_own = LossHistory()

optimizer = tf.keras.optimizers.SGD(learning_rate=0.001,momentum=0.9)

model.compile(optimizer=optimizer, loss='categorical_crossentropy',metrics=['acc',"AUC"])

model.fit(X_train,Y_train,epochs=10, validation_data=(X_test,Y_test), batch_size=16, callbacks=all_call_back)

no of ecpoch 0
learning rate 0.0010000000474974513

Epoch 00001: LearningRateScheduler reducing learning rate to 0.0009500000451225787.
Epoch 1/10
Instructions for updating:
If using Keras pass *_constraint arguments to layers.
 992/1000 [============================>.] - ETA: 0s - loss: 0.6786 - acc: 0.5735 - auc: 0.6103MICRO F1 SCORE 0.6537114603439398
auc score 0.7070491250000001

Epoch 00001: val_loss improved from inf to 0.65220, saving model to model_save\weights-01-0.6550.hdf5
1000/1000 [==============================] - 6s 6ms/step - loss: 0.6783 - acc: 0.5744 - auc: 0.6114 - val_loss: 0.6522 - val_acc: 0.6550 - val_auc: 0.7045 - lr: 9.5000e-04
no of ecpoch 1
learning rate 0.0009500000160187483

Epoch 00002: LearningRateScheduler reducing learning rate to 0.0009500000160187483.
Epoch 2/10
 994/1000 [============================>.] - ETA: 0s - loss: 0.6286 - acc: 0.6577 - auc: 0.7142MICRO F1 SCORE 0.6564675257376436
auc score 0.7220727499999999

Epoch 00002: val_loss improved fr

In [12]:
#!kill 18544

In [13]:
#%tensorboard --logdir ./logs/fit/

# Model4

In [9]:
input_layer = Input(shape=(2,))
#Dense hidden layer
model1_layer1 = Dense(20,activation='relu',kernel_initializer=tf.keras.initializers.glorot_uniform(seed=30))(input_layer)
model1_layer2 = Dense(20,activation='relu',kernel_initializer=tf.keras.initializers.glorot_uniform(seed=20))(model1_layer1)
model1_layer3 = Dense(25,activation='relu',kernel_initializer=tf.keras.initializers.glorot_uniform(seed=10))(model1_layer2)
model1_layer4 = Dense(30,activation='relu',kernel_initializer=tf.keras.initializers.glorot_uniform(seed=40))(model1_layer3)
model1_layer5= Dense(30,activation='relu',kernel_initializer=tf.keras.initializers.glorot_uniform(seed=50))(model1_layer4)
#output layer
output = Dense(2,activation='softmax',kernel_initializer=tf.keras.initializers.he_uniform(seed=50))(model1_layer5)
#Creating a model
model = Model(inputs=input_layer,outputs=output)


#Callbacks
#history_own = LossHistory()

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(optimizer=optimizer, loss='categorical_crossentropy',metrics=['acc',"AUC"])

model.fit(X_train,Y_train,epochs=10, validation_data=(X_test,Y_test), batch_size=16, callbacks=all_call_back)

no of ecpoch 0
learning rate 0.0010000000474974513

Epoch 00001: LearningRateScheduler reducing learning rate to 0.0009500000451225787.
Epoch 1/10
Instructions for updating:
If using Keras pass *_constraint arguments to layers.
 989/1000 [============================>.] - ETA: 0s - loss: 0.6348 - acc: 0.6395 - auc: 0.6914MICRO F1 SCORE 0.6453159079693017
auc score 0.7373700000000001

Epoch 00001: val_loss improved from inf to 0.61724, saving model to model_save\weights-01-0.6570.hdf5
1000/1000 [==============================] - 6s 6ms/step - loss: 0.6344 - acc: 0.6396 - auc: 0.6918 - val_loss: 0.6172 - val_acc: 0.6570 - val_auc: 0.7175 - lr: 9.5000e-04
no of ecpoch 1
learning rate 0.0009500000160187483

Epoch 00002: LearningRateScheduler reducing learning rate to 0.0009500000160187483.
Epoch 2/10
 994/1000 [============================>.] - ETA: 0s - loss: 0.6068 - acc: 0.6649 - auc: 0.7305MICRO F1 SCORE 0.6652324037782236
auc score 0.733349

Epoch 00002: val_loss improved from 0.61724

In [10]:
#!kill 3476

In [11]:
#%tensorboard --logdir ./logs4/fit/